# CostGrow Backend Comparison (WBT vs PCRaster)


## 1) Imports and Paths

In [ ]:
import os
import sys
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import rasterio
import matplotlib
import matplotlib.pyplot as plt

from fdsc.alg.costGrow import downscale_costGrow_xr
from fdsc.hp.xr import xr_to_GeoTiff
from misc.valid import build_bilinear_depth_baseline, build_metrics_table, wse_to_depth_m


: 

In [ ]:

REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / 'tests').exists() and (REPO_DIR.parent / 'tests').exists():
    REPO_DIR = REPO_DIR.parent

BASE_DIR = REPO_DIR / 'tests' / 'data' / 'bryantTechnicalNoteResolution2024'
OUT_DIR = REPO_DIR / 'misc' / '_outputs' / 'example_pcraster'
RUN_DIR = OUT_DIR / 'run_costgrow_backends'
RUN_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DEM_FINE_FP = BASE_DIR / 'dem_04m.tif'
INPUT_WSE_COARSE_FP = BASE_DIR / 'wse_32m.tif'
REFERENCE_WSE_FINE_FP = BASE_DIR / 'wse_04m.tif'

DISPLAY_SHAPE = (450, 900)

print(f"REPO_DIR:\n    {REPO_DIR}")
print(f"BASE_DIR:\n    {BASE_DIR}")
print(f"RUN_DIR:\n    {RUN_DIR}")
print(f'dem exists: {INPUT_DEM_FINE_FP.exists()}')
print(f'wse coarse exists: {INPUT_WSE_COARSE_FP.exists()}')
print(f'wse reference exists: {REFERENCE_WSE_FINE_FP.exists()}')


## 2) Version and File Diagnostics

In [ ]:
print(f'python: {sys.version.split()[0]}')
print(f'numpy: {np.__version__}')
print(f'xarray: {xr.__version__}')
print(f'rioxarray: {rioxarray.__version__}')
print(f'rasterio: {rasterio.__version__}')
print(f'matplotlib: {matplotlib.__version__}')

def file_stats(fp: Path):
    st = fp.stat()
    with rasterio.open(fp) as ds:
        return {
            'file': fp.name,
            'size_mb': round(st.st_size / (1024**2), 3),
            'shape': (ds.height, ds.width),
            'crs': str(ds.crs),
            'res': ds.res,
            'nodata': ds.nodata,
            'bounds': tuple(round(v, 3) for v in ds.bounds),
            'dtype': ds.dtypes[0],
        }

stats_l = [
    file_stats(INPUT_DEM_FINE_FP),
    file_stats(INPUT_WSE_COARSE_FP),
    file_stats(REFERENCE_WSE_FINE_FP),
]
print('input file stats:')
for d in stats_l:
    print(f"  {d['file']}: size_mb={d['size_mb']}, shape={d['shape']}, crs={d['crs']}, res={d['res']}, nodata={d['nodata']}, dtype={d['dtype']}")
print('raw file stats dicts:')
for d in stats_l:
    print(d)


## 3) Load Input Rasters

In [ ]:
def load_da(fp: Path, nodata=-9999.0) -> xr.DataArray:
    da = rioxarray.open_rasterio(fp, masked=False).squeeze().compute().rio.write_nodata(nodata)
    return da.where(da != nodata, np.nan)

dem_fine_xr = load_da(INPUT_DEM_FINE_FP)
wse_coarse_xr = load_da(INPUT_WSE_COARSE_FP)
wse_reference_xr = load_da(REFERENCE_WSE_FINE_FP)

print(f'dem_fine_xr shape={dem_fine_xr.shape}, res={dem_fine_xr.rio.resolution()}, crs={dem_fine_xr.rio.crs}')
print(f'wse_coarse_xr shape={wse_coarse_xr.shape}, res={wse_coarse_xr.rio.resolution()}, crs={wse_coarse_xr.rio.crs}')
print(f'wse_reference_xr shape={wse_reference_xr.shape}, res={wse_reference_xr.rio.resolution()}, crs={wse_reference_xr.rio.crs}')

print(f'dem wet count: {int(dem_fine_xr.notnull().sum())}')
print(f'coarse wse wet count: {int(wse_coarse_xr.notnull().sum())}')
print(f'reference wse wet count: {int(wse_reference_xr.notnull().sum())}')

## 4) Plot Setup and Helpers (inline, lightweight)

In [ ]:
cm = 1 / 2.54

plt.style.use('default')
for k, v in {
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.titlesize': 12,
    'figure.figsize': (16 * cm, 10 * cm),
    'font.family': 'serif',
}.items():
    matplotlib.rcParams[k] = v

print(f'loaded matplotlib {matplotlib.__version__}')

def nn_resample_2d(arr: np.ndarray, target_shape=DISPLAY_SHAPE) -> np.ndarray:
    assert arr.ndim == 2
    if arr.shape == target_shape:
        return arr
    yi = np.clip(np.round(np.linspace(0, arr.shape[0] - 1, target_shape[0])).astype(int), 0, arr.shape[0] - 1)
    xi = np.clip(np.round(np.linspace(0, arr.shape[1] - 1, target_shape[1])).astype(int), 0, arr.shape[1] - 1)
    return arr[np.ix_(yi, xi)]

def plot_raster(ax, da: xr.DataArray, title: str, cmap='viridis', vmin=None, vmax=None, target_shape=DISPLAY_SHAPE):
    arr = da.data.astype(float)
    arr_disp = nn_resample_2d(arr, target_shape=target_shape)

    bounds = da.rio.bounds()
    im = ax.imshow(
        arr_disp,
        extent=(bounds[0], bounds[2], bounds[1], bounds[3]),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation='nearest',
        origin='upper',
        aspect='equal',
    )

    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    raw_res = da.rio.resolution()
    ax.text(
        0.02,
        0.02,
        f'raw shape: {da.shape}\nraw res: ({raw_res[0]:.6f}, {raw_res[1]:.6f})',
        transform=ax.transAxes,
        fontsize=8,
        ha='left',
        va='bottom',
        bbox={'facecolor': 'white', 'alpha': 0.7, 'edgecolor': 'none'},
    )
    return im


## 5) Plot Inputs

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

im0 = plot_raster(axes[0, 0], dem_fine_xr, 'Input DEM fine (4 m)', cmap='copper_r')
fig.colorbar(im0, ax=axes[0, 0], shrink=0.9)

im1 = plot_raster(axes[0, 1], wse_coarse_xr, 'Input WSE coarse (32 m)', cmap='Blues')
fig.colorbar(im1, ax=axes[0, 1], shrink=0.9)

im2 = plot_raster(axes[1, 0], wse_reference_xr, 'Reference WSE fine (4 m)', cmap='Blues')
fig.colorbar(im2, ax=axes[1, 0], shrink=0.9)

wsh_reference_xr = xr.where(wse_reference_xr.notnull(), wse_reference_xr - dem_fine_xr, np.nan)
im3 = plot_raster(axes[1, 1], wsh_reference_xr, 'Reference WSH fine (4 m)', cmap='viridis')
fig.colorbar(im3, ax=axes[1, 1], shrink=0.9)

plt.show()

## 6) Run CostGrow (terrain_penalty) For Both Backends


In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logger = logging.getLogger('example_notebook')

# Execute both backends on the same terrain-penalty setup.
RUN_BACKENDS = ['wbt', 'pcraster']
run_d = {}

for backend in RUN_BACKENDS:
    backend_run_dir = RUN_DIR / backend
    backend_run_dir.mkdir(parents=True, exist_ok=True)
    output_fp = RUN_DIR / f'wse_downscaled_CostGrow_{backend}.tif'

    inference_wall_start = time.perf_counter()
    wse_downscaled_xr, meta_d = downscale_costGrow_xr(
        dem_fine_xr,
        wse_coarse_xr,
        logger=logger.getChild(backend),
        distance_fill='terrain_penalty',
        cd_backend=backend,
        write_meta=True,
        out_dir=str(backend_run_dir),
        debug=False,
    )
    inference_wall_seconds = time.perf_counter() - inference_wall_start

    xr_to_GeoTiff(wse_downscaled_xr, str(output_fp), log=logger.getChild(f'main.{backend}'))

    run_d[backend] = dict(
        output_fp=Path(output_fp),
        inference_wall_seconds=float(inference_wall_seconds),
        wse_downscaled_xr=wse_downscaled_xr,
        meta_d=meta_d,
    )

    print(f'[{backend}] downscale finished')
    print(f'[{backend}] CostGrow inference wall clock: {inference_wall_seconds:,.3f} sec')
    print(f"[{backend}] output file:\n    {output_fp}")
    print(f"[{backend}] meta entries: {len(meta_d)}")

# Keep convenient references for later cells.
wse_downscaled_xr = run_d['pcraster']['wse_downscaled_xr']
output_fp = run_d['pcraster']['output_fp']


## 7) Runtime + Accuracy Diagnostics


In [ ]:
def calc_diff_stats(pred_ar: np.ndarray, ref_ar: np.ndarray) -> dict:
    # Compute shared error stats on overlapping finite cells.
    valid = np.logical_and(np.isfinite(pred_ar), np.isfinite(ref_ar))
    cnt = int(valid.sum())
    if cnt == 0:
        return dict(overlap_cnt=0, mae=np.nan, rmse=np.nan, bias=np.nan, p95_abs=np.nan)
    diff = pred_ar[valid] - ref_ar[valid]
    return dict(
        overlap_cnt=cnt,
        mae=float(np.mean(np.abs(diff))),
        rmse=float(np.sqrt(np.mean(diff**2))),
        bias=float(np.mean(diff)),
        p95_abs=float(np.nanpercentile(np.abs(diff), 95)),
    )

summary_l = []
for backend in RUN_BACKENDS:
    output_fp = Path(run_d[backend]['output_fp'])
    assert output_fp.exists(), output_fp

    wse_backend_xr = run_d[backend]['wse_downscaled_xr'].where(np.isfinite(run_d[backend]['wse_downscaled_xr']), np.nan)
    run_d[backend]['wse_downscaled_xr'] = wse_backend_xr

    with rasterio.open(output_fp) as ds:
        print(f'[{backend}] output size MB: {output_fp.stat().st_size/(1024**2):,.3f}')
        print(f'[{backend}] output shape: {(ds.height, ds.width)}')
        print(f'[{backend}] output crs: {ds.crs}')
        print(f'[{backend}] output res: {ds.res}')
        print(f'[{backend}] output nodata: {ds.nodata}')

    stats_d = calc_diff_stats(wse_backend_xr.data, wse_reference_xr.data)
    stats_d.update(
        backend=backend,
        runtime_sec=float(run_d[backend]['inference_wall_seconds']),
        wet_cnt=int(wse_backend_xr.notnull().sum()),
    )
    summary_l.append(stats_d)

summary_df = pd.DataFrame(summary_l).set_index('backend').sort_index()
print('Runtime + WSE accuracy vs reference')
summary_df.round(6)

wbt_row = summary_df.loc['wbt']
pcr_row = summary_df.loc['pcraster']
runtime_ratio = float(pcr_row['runtime_sec'] / wbt_row['runtime_sec'])
runtime_delta = float(pcr_row['runtime_sec'] - wbt_row['runtime_sec'])

print(f"PCRaster runtime ratio vs WBT: {runtime_ratio:.3f}x")
print(f"PCRaster runtime delta vs WBT: {runtime_delta:+,.3f} sec")

delta_cols = ['mae', 'rmse', 'bias', 'p95_abs', 'wet_cnt']
delta_d = {k: float(pcr_row[k] - wbt_row[k]) for k in delta_cols}
print('PCRaster minus WBT (vs reference):')
for k in delta_cols:
    print(f'  {k}: {delta_d[k]:+,.6f}')

wse_wbt_xr = run_d['wbt']['wse_downscaled_xr']
wse_pcraster_xr = run_d['pcraster']['wse_downscaled_xr']
backend_diff_stats = calc_diff_stats(wse_pcraster_xr.data, wse_wbt_xr.data)
print('PCRaster vs WBT direct WSE difference stats')
for k in ['overlap_cnt', 'mae', 'rmse', 'bias', 'p95_abs']:
    val = backend_diff_stats[k]
    if isinstance(val, float):
        print(f'  {k}: {val:,.6f}')
    else:
        print(f'  {k}: {val}')

# Keep plot variables aligned with the notebook plotting cell.
wse_downscaled_xr = wse_pcraster_xr
diff_xr = wse_downscaled_xr - wse_reference_xr
diff_vs_wbt_xr = wse_pcraster_xr - wse_wbt_xr


## 8) Plot WBT vs PCRaster Outputs


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

im0 = plot_raster(axes[0, 0], run_d['wbt']['wse_downscaled_xr'], 'Output WSE downscaled (WBT)', cmap='Blues')
fig.colorbar(im0, ax=axes[0, 0], shrink=0.9)

im1 = plot_raster(axes[0, 1], run_d['pcraster']['wse_downscaled_xr'], 'Output WSE downscaled (PCRaster)', cmap='Blues')
fig.colorbar(im1, ax=axes[0, 1], shrink=0.9)

im2 = plot_raster(axes[1, 0], wse_reference_xr, 'Reference WSE fine', cmap='Blues')
fig.colorbar(im2, ax=axes[1, 0], shrink=0.9)

vmax = float(np.nanpercentile(np.abs(diff_vs_wbt_xr.data), 99))
im3 = plot_raster(axes[1, 1], diff_vs_wbt_xr, 'PCRaster - WBT WSE', cmap='coolwarm', vmin=-vmax, vmax=vmax)
fig.colorbar(im3, ax=axes[1, 1], shrink=0.9)

plt.show()


## 9) Artifacts

- Output WSE (WBT): `misc/_outputs/example_pcraster/run_costgrow_backends/wse_downscaled_CostGrow_wbt.tif`
- Output WSE (PCRaster): `misc/_outputs/example_pcraster/run_costgrow_backends/wse_downscaled_CostGrow_pcraster.tif`
- Additional debug rasters/logs are under `misc/_outputs/example_pcraster/run_costgrow_backends/`
